# Dune types with ibis-dune

Paid Trino access is required. Export `DUNE_API_KEY` before running this notebook.

`iq_protocol_polygon.enterprise_evt_rented` is a small static table (about 6–7k rows). Every execute below is filtered or limited.

- **uint256** stays a full integer (not float64). In the schema it is `decimal(78, 0)`; the rich preview labels it `uint256`.
- **`contract_address`** from `table()` is binary.
- **`hex_literal`**, **`raw_predicate`**, and **`raw_scalar`** compile Dune SQL fragments Ibis cannot express on its own.


In [1]:
import os

import ibis
from ibis import _
from ibis_dune.ops import hex_literal, raw_predicate, raw_scalar

api_key = os.environ.get("DUNE_API_KEY")
if not api_key:
    raise RuntimeError("Set DUNE_API_KEY to a paid Dune key with Trino access")
con = ibis.dune.connect(dune_api_key=api_key)

## One row: uint256 and binary

`hex_literal` matches the known contract and transaction so the scan is a single row. The schema dtypes come from `table()`.


In [2]:
contract = "0xbf9f6b1d910aa207daa400931430ef110570f8ff"
tx_hash = "0x06c4bf3d702b2adbadb52230a2d1c507da55b2ef07b285fe3a4d55f8daf475be"
t = con.table("enterprise_evt_rented", database="iq_protocol_polygon")
one = (
    t.filter(
        _.contract_address == hex_literal(contract),
        _.evt_tx_hash == hex_literal(tx_hash),
    )
    .select("evt_block_number", "evt_index", "contract_address", "rentaltokenid")
    .order_by("evt_block_number", "evt_index")
)
one.schema()["rentaltokenid"], one.schema()["contract_address"]

(Decimal(precision=78, scale=0, nullable=True), Binary(nullable=True))

In [3]:
row = one.execute()
token_id = row["rentaltokenid"].iloc[0]
address = row["contract_address"].iloc[0]
type(token_id), token_id, type(address), address

(decimal.Decimal,
 Decimal('115597092877761069019903437234573841225780679087707680554709867281103459621204'),
 str,
 '0xbf9f6b1d910aa207daa400931430ef110570f8ff')

With interactive mode on, the table preview labels `rentaltokenid` as `uint256` and renders `contract_address` as hex.


In [4]:
ibis.options.interactive = True
one

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ evt_block_number ┃ evt_index ┃ contract_address                           ┃ rentaltokenid                                                                  ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ int64            │ int64     │ binary                                     │ uint256                                                                        │
├──────────────────┼───────────┼────────────────────────────────────────────┼────────────────────────────────────────────────────────────────────────────────┤
│         69349932 │       181 │ 0xbf9f6b1d910aa207daa400931430ef110570f8ff │ 115597092877761069019903437234573841225780679087707680554709867281103459621204 │
└──────────────────┴───────────┴────────────────────────────────────────────┴────────────────────────────────────────────────────────────────────────────────┘

## `raw_predicate`

A verbatim Trino boolean, still limited to two rows.


In [5]:
(
    t.filter(raw_predicate(f"contract_address = {contract}"))
    .select("evt_block_number", "evt_index", "rentaltokenid")
    .order_by("evt_block_number", "evt_index")
    .limit(2)
    .execute()
)

,evt_block_number,evt_index,rentaltokenid
0,23440465,219,6657404521948493445777574718502674526780122234...
1,23450550,369,5135697478176795391562593291404386300544142432...


## `raw_scalar`

A verbatim SQL expression with an explicit ibis dtype, on the same row.


In [6]:
one.select(
    "evt_block_number",
    probe=raw_scalar("CAST(42 AS BIGINT)", "int64"),
).execute()

,evt_block_number,probe
0,69349932,42
